In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 設定資料路徑'/content/drive/MyDrive/金融資料探勘/2021'
folder_path = '/content/drive/MyDrive/金融資料探勘/2021'

In [3]:
import pandas as pd
import os
from google.colab import drive

# 1. 設定資料夾路徑
input_file = os.path.join(folder_path, '2021 原始資料.xlsx') # 確認是 .xlsx 檔案
output_file = os.path.join(folder_path, 'Path_2021_Full.xlsx')

# 2. 讀取資料
if os.path.exists(input_file):
    # 根據檔案類型使用正確的讀取函數
    if input_file.endswith('.csv'):
        df = pd.read_csv(input_file, encoding='utf-8') # 若有亂碼可嘗試 'big5' 或 'latin1'
    elif input_file.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(input_file)
    else:
        print(f"錯誤：不支援的檔案格式 '{input_file}'。請提供 .csv 或 .xlsx 檔案。")
        df = None # 確保 df 在此情況下為 None

    if df is not None:
        # 3. 資料轉換邏輯
        # 將 '年月日' 轉換為 datetime 格式
        df['Date_dt'] = pd.to_datetime(df['年月日'])

        # 新增 File 欄位：格式為 OptionsDaily_YYYY_MM_DD.csv
        df['File'] = df['Date_dt'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

        # 重新命名與排序欄位
        df_final = df.rename(columns={'年月日': 'Date', '收盤價(元)': 'S0'})
        df_final = df_final[['Date', 'File', 'S0']]

        # 4. 儲存結果回雲端硬碟
        df_final.to_excel(output_file, index=False)

        print(f"\n處理完成！檔案已儲存至雲端硬碟：{output_file}")

        # 預覽結果
        print("\n資料預覽：")
        print(df_final.head())
else:
    print(f"錯誤：在路徑 {input_file} 找不到檔案。請確認檔案已上傳至該資料夾。")


處理完成！檔案已儲存至雲端硬碟：/content/drive/MyDrive/金融資料探勘/2021/Path_2021_Full.xlsx

資料預覽：
        Date                         File        S0
0 2021-01-04  OptionsDaily_2021_01_04.csv  14902.03
1 2021-01-05  OptionsDaily_2021_01_05.csv  15000.03
2 2021-01-06  OptionsDaily_2021_01_06.csv  14983.13
3 2021-01-07  OptionsDaily_2021_01_07.csv  15214.00
4 2021-01-08  OptionsDaily_2021_01_08.csv  15463.95


In [4]:
import pandas as pd
import os
from google.colab import drive

# 1. 定義檔案名稱
price_file = os.path.join(folder_path, 'Path_2021_Full.xlsx')
settle_file = os.path.join(folder_path, '2021 臺指選擇權.xlsx') # 修正檔案名稱

# 2. 讀取資料
path_df = pd.read_excel(price_file)
path_df['Date'] = pd.to_datetime(path_df['Date'])

settle_df = pd.read_excel(settle_file) # 修正讀取方式為 read_excel
settle_df['最後結算日'] = pd.to_datetime(settle_df['最後結算日'])

# 篩選「月選擇權」(排除契約月份含 'W' 者)
month_settle = settle_df[~settle_df['契約月份'].str.contains('W', na=False)].copy() # 增加 na=False 處理 NaN 值
month_settle = month_settle.sort_values('最後結算日')

# 3. 核心邏輯：匹配合約與計算 Maturity (條件: 距離結算日 >= 1)
results = []

for _, row in path_df.iterrows():
    curr_date = row['Date']

    # 尋找結算日 >= 當前日期 + 1 天的合約
    mask = (month_settle['最後結算日'] - curr_date).dt.days >= 1
    valid_contracts = month_settle[mask]

    if not valid_contracts.empty:
        target = valid_contracts.iloc[0]
        expiry_date = target['最後結算日']
        maturity = (expiry_date - curr_date).days

        results.append({
            'Date': curr_date.strftime('%Y/%m/%d'),
            'File': row['File'],
            'S0': row['S0'],
            'Contract': target['契約月份'],
            'ContractExpiryDate': expiry_date.strftime('%Y/%m/%d'),
            'Maturity': maturity
        })

# 4. 轉換為 DataFrame 並儲存回雲端硬碟
df_final = pd.DataFrame(results)
output_path = os.path.join(folder_path, 'Path_2021_Full_Calculated.xlsx')
df_final.to_excel(output_path, index=False)

print(f"\n處理完成！檔案已儲存至雲端硬碟：{output_path}")


處理完成！檔案已儲存至雲端硬碟：/content/drive/MyDrive/金融資料探勘/2021/Path_2021_Full_Calculated.xlsx


In [5]:
import pandas as pd
import os
from google.colab import drive

# 1. 設定路徑
file_name = 'Path_2021_Full_Calculated.xlsx'
full_path = os.path.join(folder_path, file_name)

# 2. 讀取 Excel 檔案
if os.path.exists(full_path):
    # 使用 pandas 讀取 Excel
    df = pd.read_excel(full_path)

    # 3. 新增 Rf 欄位
    # 2021 年台銀一年期定儲一般機動利率固定為 0.84%
    # 在金融計算中，通常以小數形式表示，即 0.0084
    df['Rf'] = 0.0084

    # 4. 儲存檔案
    # 您可以選擇覆蓋原檔，或另存新檔（建議另存以保備份）
    output_path = os.path.join(folder_path, 'Path_2021_Final_with_Rf.xlsx')
    df.to_excel(output_path, index=False)

    print(f"處理完成！")
    print(f"已新增 Rf 欄位（數值：0.0084），檔案儲存至：{output_path}")

    # 顯示前 5 筆預覽
    print("\n資料預覽：")
    print(df[['Date', 'S0', 'Rf']].head())
else:
    print(f"錯誤：在路徑 {full_path} 找不到檔案。請確認檔名與資料夾路徑正確。")

處理完成！
已新增 Rf 欄位（數值：0.0084），檔案儲存至：/content/drive/MyDrive/金融資料探勘/2021/Path_2021_Final_with_Rf.xlsx

資料預覽：
         Date        S0      Rf
0  2021/01/04  14902.03  0.0084
1  2021/01/05  15000.03  0.0084
2  2021/01/06  14983.13  0.0084
3  2021/01/07  15214.00  0.0084
4  2021/01/08  15463.95  0.0084
